# Cover–Stego Separability as a Predictor of Steganalysis Error

A source-disjoint evaluation with negative controls on BOSSbase 1.01.

**Questions**

1. Do separability measures computed on training covers (Fisher ratio, mean shift, deflection, MMD, Jensen–Shannon divergence) rank embedding conditions in the same order as the error of trained detectors on unseen covers?
2. How do detection estimates change between a single-cover design (P0), a row-level split (P1), a source-disjoint split (P2) and a shifted cover source (P3)?
3. Payload whitening of a uniform message cannot change the stego distribution. How often does each protocol nevertheless report a whitening effect, and does it detect the real effect that exists for a biased message?
4. How does the uncertainty of an AUC estimate depend on the number of independent test covers?

**Embedding schemes**: LSB matching with uniform positions (LSBM) and with activity-weighted positions (LSBM-A), HILL and S-UNIWARD (simulated at the payload–distortion bound), at 0.05–0.40 bpp.
**Detectors**: logistic regression and a random forest on 13 summary features, an ensemble of Fisher discriminants on SPAM-686 features, and SRNet.

**How to run**

- Set `RUN_MODE = "quick"` in the first code cell for a short pipeline test on sample images (a few minutes, results stamped as test output), then `RUN_MODE = "full"` for the reported experiments.
- BOSSbase 1.01 (`*.pgm`) is read from `DATA_ROOT/BOSSbase_1.01`. If it is absent, the notebook looks for `BOSSbase_1.01.zip` and otherwise tries to download it.
- Large intermediate files (covers, stego sets, features, checkpoints) go to `WORK_DIR`; small result files (tables, figures, CSV, `results.json`, `outputs_summary.txt`) go to `OUT_DIR`. On a cluster the three locations can be set with the environment variables `CSS_DATA_ROOT`, `CSS_WORK_DIR` and `CSS_OUT_DIR`.
- Every stage is cached. If the session ends, run all cells again: finished stages are skipped and SRNet training resumes from its last checkpoint.


In [ ]:
# Imports, paths and configuration
import os, sys, json, time, math, zipfile, warnings, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import stats
from scipy.signal import fftconvolve
from scipy.ndimage import correlate, uniform_filter
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from joblib import Parallel, delayed

PROJECT_NAME = "Cover_Stego_Separability"

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/Datasets")            # read only
    WORK_DIR = Path("/content/work") / PROJECT_NAME                # large caches, local disk
    OUT_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME  # small result files only
else:
    _home = Path.home()
    _scratch = Path(os.environ.get("SCRATCH", _home / "scratch"))
    DATA_ROOT = Path(os.environ.get("CSS_DATA_ROOT", _scratch / "Datasets"))
    WORK_DIR = Path(os.environ.get("CSS_WORK_DIR", _scratch / "work" / PROJECT_NAME))
    OUT_DIR = Path(os.environ.get("CSS_OUT_DIR", _home / "Outputs" / PROJECT_NAME))

RUN_MODE = "full"     # "full" for the reported experiments; "quick" for a short pipeline test on sample images
QUICK = RUN_MODE == "quick" or os.environ.get("CSS_QUICK", "0") == "1"

def _allocated_cores():
    if os.environ.get("SLURM_CPUS_PER_TASK"):
        return int(os.environ["SLURM_CPUS_PER_TASK"])
    try:
        return len(os.sched_getaffinity(0))
    except Exception:
        return os.cpu_count() or 2

CONFIG = {
    "seed": 2026,
    "image_size": 256,
    "n_train": 4000, "n_val": 1000, "n_test": 5000,       # covers; each cover has one stego per condition
    "payloads_bpp": [0.05, 0.10, 0.20, 0.30, 0.40],
    "embedders": ["LSBM", "LSBM-A", "HILL", "S-UNIWARD"],
    "boss_dirname": "BOSSbase_1.01",
    "boss_url": "http://dde.binghamton.edu/download/ImageDB/BOSSbase_1.01.zip",
    "n_jobs": int(os.environ.get("CSS_N_JOBS", max(1, _allocated_cores() - 1))),
    # detectors
    "rf": {"n_estimators": 120, "max_depth": 4, "min_samples_leaf": 4},
    "fld_ensemble": {"n_learners": 51, "d_sub": 200, "ridge": 1e-6},
    "srnet": {"enabled": True, "batch_pairs": 16, "lr": 1e-3, "weight_decay": 2e-4,
              "max_epochs_scratch": 400, "max_epochs_finetune": 150,
              "min_delta": 0.002, "patience_lr": 20, "patience_stop": 20, "warmup_epochs": 80,
              "payloads_bpp": [0.40, 0.30, 0.20, 0.10, 0.05]},   # curriculum, high to low
    # statistics
    "n_boot": 2000, "ci_level": 0.95,
    "ntest_grid": [11, 50, 200, 1000, 5000],
    # legacy single-cover protocol (P0), kept exactly as in the earlier design
    "p0": {"n_variants": 36, "test_size": 0.30, "jitter_sigma": 0.003,
           "alpha_grid": [0.0, 0.25, 0.5, 0.75, 1.0], "n_replicates": 200, "n_source_covers": 20},
    # whitening control
    "control": {"payload_bpp": 0.20, "alphas": [0.0, 0.5, 1.0], "bit_bias": [0.5, 0.75]},
    # separability measures
    "mmd_subsample": 2000, "shrinkage": 1e-3,
}
if QUICK:
    CONFIG.update({"image_size": 96, "n_train": 120, "n_val": 40, "n_test": 120,
                   "payloads_bpp": [0.10, 0.40], "n_boot": 200, "ntest_grid": [11, 50, 120],
                   "mmd_subsample": 200})
    CONFIG["fld_ensemble"].update({"n_learners": 11, "d_sub": 60})
    CONFIG["p0"].update({"n_replicates": 12, "n_source_covers": 3})
    CONFIG["srnet"].update({"warmup_epochs": 0, "max_epochs_scratch": 2, "max_epochs_finetune": 1,
                            "payloads_bpp": [0.40, 0.10], "batch_pairs": 4})
    OUT_DIR = OUT_DIR / "QUICK_TEST"
    WORK_DIR = WORK_DIR / "QUICK_TEST"

FIG_DIR, TABLE_DIR = OUT_DIR / "Figures", OUT_DIR / "Tables"
for p in [WORK_DIR, OUT_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
SUMMARY = OUT_DIR / "outputs_summary.txt"
RESULTS = {}

def log(msg=""):
    line = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open(SUMMARY, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def save_results():
    with open(OUT_DIR / "results.json", "w", encoding="utf-8") as f:
        json.dump(RESULTS, f, indent=2, default=float)

def cond_key(emb, payload):
    return f"{emb}@{payload:.2f}"

N_TOTAL = CONFIG["n_train"] + CONFIG["n_val"] + CONFIG["n_test"]
log(f"Project {PROJECT_NAME} | QUICK={QUICK} | covers={N_TOTAL} | jobs={CONFIG['n_jobs']}")
log(f"DATA_ROOT={DATA_ROOT}\nWORK_DIR={WORK_DIR}\nOUT_DIR={OUT_DIR}")

## 1. Cover images

BOSSbase 1.01 images are downsampled to 256×256 (bicubic). Covers are split **by source image** into training, validation and test partitions. A second version of the test covers, cut as central 256×256 crops from the same source images, provides a shifted cover source: same scenes and cameras, different processing.

In [ ]:
# Cover preparation: BOSSbase 1.01 -> 256x256 covers, plus a processing-shifted test set
def _download(url, dst):
    log(f"Downloading {url}")
    t0 = time.time()
    def hook(k, bs, total):
        if k % 2000 == 0 and total > 0:
            print(f"  {100 * k * bs / total:5.1f}%  ({time.time() - t0:.0f}s)", flush=True)
    urllib.request.urlretrieve(url, dst, hook)

def locate_boss():
    d = DATA_ROOT / CONFIG["boss_dirname"]
    files = sorted(d.rglob("*.pgm")) if d.exists() else []
    if len(files) >= N_TOTAL:
        return files
    z = DATA_ROOT / (CONFIG["boss_dirname"] + ".zip")
    target = (WORK_DIR if IN_COLAB else DATA_ROOT) / CONFIG["boss_dirname"]
    if not z.exists():
        z = WORK_DIR / (CONFIG["boss_dirname"] + ".zip")
        if not z.exists():
            try:
                _download(CONFIG["boss_url"], z)
            except Exception as e:
                raise FileNotFoundError(
                    f"BOSSbase not found under {d} and download failed ({e}). "
                    f"Place the .pgm files in {d} (or the zip in {DATA_ROOT}) and rerun this cell.")
    log(f"Extracting {z} -> {target}")
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(target)
    return sorted(target.rglob("*.pgm"))

def surrogate_covers(n, size, rng):
    """Tiles of scikit-image sample photographs. Pipeline testing only."""
    from skimage import data as skd
    imgs = [skd.camera(), skd.moon(), skd.coins(), skd.brick(), skd.grass(), skd.gravel()]
    out = np.empty((n, size, size), np.uint8)
    for i in range(n):
        im = imgs[i % len(imgs)]
        r = rng.integers(0, im.shape[0] - 2 * size); c = rng.integers(0, im.shape[1] - 2 * size)
        out[i] = np.asarray(Image.fromarray(im[r:r + 2 * size, c:c + 2 * size]).resize((size, size), Image.BICUBIC))
    return out

def prepare_covers():
    S = CONFIG["image_size"]
    f_main, f_shift, f_split = WORK_DIR / "covers.npy", WORK_DIR / "covers_shift.npy", WORK_DIR / "split.json"
    if f_main.exists() and f_shift.exists() and f_split.exists():
        log("Covers already prepared; loading cache.")
        return np.load(f_main, mmap_mode="r"), np.load(f_shift, mmap_mode="r"), json.load(open(f_split))
    rng = np.random.default_rng(CONFIG["seed"])
    perm = rng.permutation(N_TOTAL)
    split = {"train": sorted(perm[:CONFIG["n_train"]].tolist()),
             "val": sorted(perm[CONFIG["n_train"]:CONFIG["n_train"] + CONFIG["n_val"]].tolist()),
             "test": sorted(perm[CONFIG["n_train"] + CONFIG["n_val"]:].tolist())}
    if QUICK:
        log("QUICK mode: surrogate covers from scikit-image samples (not BOSSbase).")
        covers = surrogate_covers(N_TOTAL, S, rng)
        shift = np.stack([np.rot90(covers[i]).copy() for i in split["test"]])  # stand-in only
        split["source"] = "surrogate"
    else:
        files = locate_boss()[:N_TOTAL]
        covers = np.empty((N_TOTAL, S, S), np.uint8)
        shift = np.empty((len(split["test"]), S, S), np.uint8)
        test_pos = {i: k for k, i in enumerate(split["test"])}
        t0 = time.time()
        for i, f in enumerate(files):
            im = Image.open(f).convert("L")
            covers[i] = np.asarray(im.resize((S, S), Image.BICUBIC))          # main source: bicubic downsampling
            if i in test_pos:                                                 # shifted source: central crop
                w, h = im.size; l, t = (w - S) // 2, (h - S) // 2
                shift[test_pos[i]] = np.asarray(im.crop((l, t, l + S, t + S)))
            if (i + 1) % 1000 == 0:
                log(f"  covers {i + 1}/{N_TOTAL} ({time.time() - t0:.0f}s)")
        split["source"] = "BOSSbase_1.01"
        split["files"] = [f.name for f in files]
    np.save(f_main, covers); np.save(f_shift, shift)
    json.dump(split, open(f_split, "w"))
    return np.load(f_main, mmap_mode="r"), np.load(f_shift, mmap_mode="r"), split

COVERS, COVERS_SHIFT, SPLIT = prepare_covers()
IDX = {k: np.array(SPLIT[k]) for k in ["train", "val", "test"]}
assert not (set(SPLIT["train"]) & set(SPLIT["test"])) and not (set(SPLIT["val"]) & set(SPLIT["test"]))
log(f"Covers {COVERS.shape}, shifted test covers {COVERS_SHIFT.shape}, source={SPLIT['source']}")
RESULTS["data"] = {"source": SPLIT["source"], "n_covers": int(N_TOTAL), "image_size": CONFIG["image_size"],
                   **{f"n_{k}": int(len(IDX[k])) for k in IDX}}

## 2. Embedding schemes

LSB matching embeds actual message bits. HILL and S-UNIWARD are simulated as a payload-limited sender at the rate–distortion bound of their additive cost. Self-tests check the simulator's change rate, the S-UNIWARD cost against a brute-force evaluation of its definition, and the LSB-matching change rate.

In [ ]:
# Embedding: LSB matching with real message bits, and cost-based simulated embedding
WET = 1e8

def activity_positions(img, n, rng, keep_prob=0.62, jitter=0.20, shuffle=0.35):
    """Activity-weighted position selection of the earlier design (its 'hybrid' strategy)."""
    f = img.astype(np.float32)
    gx = np.zeros_like(f); gy = np.zeros_like(f)
    gx[:, 1:] = np.abs(f[:, 1:] - f[:, :-1]); gy[1:, :] = np.abs(f[1:, :] - f[:-1, :])
    act = gx + gy
    if act.max() > 0:
        act /= act.max()
    mask = rng.binomial(1, keep_prob, size=img.shape)
    score = 0.40 * mask + 0.40 * np.sqrt(act) + jitter * rng.random(img.shape)
    elig = np.flatnonzero(mask.ravel() == 1)
    if elig.size < n:
        elig = np.arange(img.size)
    ranked = elig[np.argsort(score.ravel()[elig])[::-1]]
    pool = ranked[:min(ranked.size, max(n, int(n * (1.0 + shuffle))))]
    return rng.choice(pool, size=n, replace=False)

def lsbm_embed(img, payload, rng, adaptive=False, alpha=0.0, bit_bias=0.5):
    """LSB matching (+-1 embedding) of n = payload * pixels message bits.
    bit_bias = P(bit = 1) of the message source; alpha = fraction of bits XOR-masked ("whitening")."""
    flat = img.astype(np.int16).ravel().copy()
    n = max(1, int(payload * flat.size))
    bits = (rng.random(n) < bit_bias).astype(np.int16)
    if alpha > 0:
        mask = rng.integers(0, 2, n, dtype=np.int16) & (rng.random(n) < alpha).astype(np.int16)
        bits ^= mask
    pos = activity_positions(img, n, rng) if adaptive else rng.choice(flat.size, size=n, replace=False)
    v = flat[pos]
    change = (v & 1) != bits
    step = np.where(rng.random(n) < 0.5, -1, 1).astype(np.int16)
    step[v == 0] = 1; step[v == 255] = -1
    flat[pos] = v + step * change
    return flat.reshape(img.shape).astype(np.uint8)

_KB = np.array([[-1, 2, -1], [2, -4, 2], [-1, 2, -1]], np.float64)

def hill_cost(img):
    x = img.astype(np.float64)
    r = np.abs(correlate(x, _KB, mode="reflect"))
    xi = 1.0 / (uniform_filter(r, 3, mode="reflect") + 1e-10)
    return uniform_filter(xi, 15, mode="reflect")

def _uniward_filters():
    import pywt
    w = pywt.Wavelet("db8")
    lo, hi = np.array(w.dec_lo), np.array(w.dec_hi)
    return [np.outer(lo, hi), np.outer(hi, lo), np.outer(hi, hi)]
_UF = None

def suniward_cost(img, sigma=1.0):
    """S-UNIWARD additive approximation: cost of a +-1 change at each pixel,
    rho_ij = sum_k sum_uv |F_k[u - i, v - j]| / (sigma + |W_k(X)_uv|)."""
    global _UF
    if _UF is None:
        _UF = _uniward_filters()
    L = _UF[0].shape[0]
    xp = np.pad(img.astype(np.float64), L, mode="symmetric")
    rho = np.zeros_like(xp)
    for F in _UF:
        R = fftconvolve(xp, F[::-1, ::-1], mode="valid")           # correlation: R[u,v] = sum X[u+a, v+b] F[a,b]
        rho += fftconvolve(1.0 / (sigma + np.abs(R)), np.abs(F), mode="full")
    return rho[L:-L, L:-L]

def _h3(pp, pm):
    p0 = 1.0 - pp - pm
    out = 0.0
    for p in (pp, pm, p0):
        q = p[p > 1e-300]
        out -= float(np.sum(q * np.log2(q)))
    return out

def simulate_ternary(img, cost, payload, rng):
    """Payload-limited sender simulated at the rate-distortion bound for an additive cost."""
    x = img.astype(np.int16)
    rp = np.where(x == 255, WET, cost); rm = np.where(x == 0, WET, cost)
    target = payload * x.size
    def probs(lam):
        ep, em = np.exp(-np.minimum(lam * rp, 700)), np.exp(-np.minimum(lam * rm, 700))
        z = 1.0 + ep + em
        return ep / z, em / z
    lo, hi = 0.0, 1.0 / max(np.median(cost), 1e-12)
    while _h3(*probs(hi)) > target and hi < 1e30:
        hi *= 2.0
    for _ in range(40):
        mid = 0.5 * (lo + hi)
        if _h3(*probs(mid)) > target:
            lo = mid
        else:
            hi = mid
    pp, pm = probs(0.5 * (lo + hi))
    u = rng.random(x.shape)
    y = x + (u < pp).astype(np.int16) - ((u >= pp) & (u < pp + pm)).astype(np.int16)
    return y.astype(np.uint8)

def embed(img, embedder, payload, seed, **kw):
    rng = np.random.default_rng(seed)
    if embedder == "LSBM":
        return lsbm_embed(img, payload, rng, adaptive=False, **kw)
    if embedder == "LSBM-A":
        return lsbm_embed(img, payload, rng, adaptive=True, **kw)
    if embedder == "HILL":
        return simulate_ternary(img, hill_cost(img), payload, rng)
    if embedder == "S-UNIWARD":
        return simulate_ternary(img, suniward_cost(img), payload, rng)
    raise ValueError(embedder)

def _selftest_embedding():
    rng = np.random.default_rng(0)
    img = np.asarray(COVERS[0]); n = img.size
    # (a) cost-based simulator hits the target entropy and changes by at most 1
    for name, cost in [("HILL", hill_cost(img)), ("S-UNIWARD", suniward_cost(img))]:
        y = simulate_ternary(img, cost, 0.4, rng)
        d = y.astype(int) - img.astype(int)
        assert np.abs(d).max() <= 1, name
        rate = np.mean(d != 0)
        assert 0.01 < rate < 0.2, (name, rate)
    # (b) S-UNIWARD cost equals the brute-force distortion of a single +1 change
    global _UF
    L = _UF[0].shape[0]
    xp = np.pad(img.astype(np.float64), L, mode="symmetric")
    Rs = [fftconvolve(xp, F[::-1, ::-1], mode="valid") for F in _UF]
    rho = suniward_cost(img)
    for (i, j) in [(5, 7), (img.shape[0] // 2, img.shape[1] // 2), (img.shape[0] - 3, 2)]:
        e = np.zeros_like(xp); e[i + L, j + L] = 1.0
        brute = sum(np.sum(np.abs(fftconvolve(e, F[::-1, ::-1], mode="valid")) / (1.0 + np.abs(R)))
                    for F, R in zip(_UF, Rs))
        assert abs(brute - rho[i, j]) < 1e-6 * max(1.0, brute), (i, j, brute, rho[i, j])
    # (c) LSB matching: about half of the used pixels change, each by exactly 1
    y = lsbm_embed(img, 0.4, rng)
    d = y.astype(int) - img.astype(int)
    assert np.abs(d).max() == 1 and abs(np.mean(d != 0) - 0.2) < 0.02
    log("Embedding self-tests passed (simulator rate, S-UNIWARD cost vs brute force, LSBM change rate).")
_selftest_embedding()

## 3. Stego sets

One stego image per cover and condition, with a seed that depends only on the cover index and the condition. The whitening-control sets use LSB matching at one payload with a uniform message (`bit_bias = 0.5`, the null case) and a biased message (`bit_bias = 0.75`, the positive control).

In [ ]:
# Stego sets (cached on disk): one stego per cover and condition, seeded per (cover, condition)
def _stable_tag(s):
    import hashlib
    return int(hashlib.md5(s.encode()).hexdigest()[:8], 16)

def stego_set(covers, name, embedder, payload, **kw):
    """Returns a uint8 memmap aligned with `covers`. Generated once, then reused."""
    f = WORK_DIR / f"stego_{name}.npy"
    if f.exists():
        return np.load(f, mmap_mode="r")
    tag = _stable_tag(name)
    t0 = time.time()
    n = len(covers)
    out = np.lib.format.open_memmap(f.with_suffix(".tmp.npy"), mode="w+", dtype=np.uint8, shape=covers.shape)
    chunk = 500
    for s in range(0, n, chunk):
        e = min(n, s + chunk)
        res = Parallel(n_jobs=CONFIG["n_jobs"])(
            delayed(embed)(np.asarray(covers[i]), embedder, payload, [CONFIG["seed"], i, tag], **kw)
            for i in range(s, e))
        out[s:e] = np.stack(res)
        if e % 2000 == 0 or e == n:
            log(f"  {name}: {e}/{n} ({time.time() - t0:.0f}s)")
    out.flush(); del out
    os.replace(f.with_suffix(".tmp.npy"), f)
    return np.load(f, mmap_mode="r")

CONDITIONS = [(e, p) for e in CONFIG["embedders"] for p in CONFIG["payloads_bpp"]]
STEGO, STEGO_SHIFT = {}, {}
log(f"Generating {len(CONDITIONS)} stego sets on the main source and on the shifted test source")
for emb, p in CONDITIONS:
    k = cond_key(emb, p)
    STEGO[k] = stego_set(COVERS, k, emb, p)
    STEGO_SHIFT[k] = stego_set(COVERS_SHIFT, "shift_" + k, emb, p)

# Whitening control: LSB matching with real message bits. bit_bias = 0.5 is the null case
# (uniform message: masking cannot change the stego distribution); bit_bias != 0.5 is the positive control.
CTRL = {}
cp = CONFIG["control"]["payload_bpp"]
for q in CONFIG["control"]["bit_bias"]:
    for a in CONFIG["control"]["alphas"]:
        name = f"ctrl_q{q:.2f}_a{a:.2f}"
        CTRL[(q, a)] = stego_set(COVERS, name, "LSBM", cp, alpha=a, bit_bias=q)
rate = {k: float(np.mean(np.asarray(STEGO[k][:50]) != np.asarray(COVERS[:50]))) for k in STEGO}
log("Change rates (first 50 covers): " + ", ".join(f"{k}={v:.3f}" for k, v in rate.items()))
RESULTS["change_rate"] = rate

## 4. Features

`F13`: thirteen summary statistics of intensity, gradients and the LSB plane. `SPAM`: second-order subtractive pixel adjacency features, 686 dimensions (T = 3).

In [ ]:
# Feature representations: the 13 summary features of the earlier design, and SPAM-686
FEAT13_NAMES = ["mean", "std", "median", "q25", "q75", "grad_x_mean", "grad_y_mean", "grad_diag_mean",
                "grad_x_std", "grad_y_std", "lsb_mean", "lsb_transition_x", "lsb_transition_y"]

def feat13(img):
    x = img.astype(np.float32)
    rd, cd = np.abs(np.diff(x, axis=1)), np.abs(np.diff(x, axis=0))
    dd = np.abs(x[1:, 1:] - x[:-1, :-1])
    lsb = (img & 1).astype(np.float32)
    return np.array([x.mean(), x.std(), np.median(x), np.percentile(x, 25), np.percentile(x, 75),
                     rd.mean(), cd.mean(), dd.mean(), rd.std(), cd.std(), lsb.mean(),
                     np.mean(lsb[:, 1:] != lsb[:, :-1]), np.mean(lsb[1:, :] != lsb[:-1, :])], np.float64)

def _spam_dir(x, dy, dx, T=3):
    """Second-order transition probabilities P(D3 | D2, D1) of pixel differences along (dy, dx)."""
    H, W = x.shape
    def view(k):  # pixel k steps along the direction; all four positions of a triple fit inside the image
        r0 = max(0, -3 * dy) + k * dy; c0 = max(0, -3 * dx) + k * dx
        return x[r0:r0 + H - 3 * abs(dy), c0:c0 + W - 3 * abs(dx)]
    p = [view(k) for k in range(4)]
    d1, d2, d3 = p[0] - p[1], p[1] - p[2], p[2] - p[3]
    ok2 = (np.abs(d1) <= T) & (np.abs(d2) <= T)
    ok3 = ok2 & (np.abs(d3) <= T)
    B = 2 * T + 1
    pair = np.bincount(((d1[ok2] + T) * B + (d2[ok2] + T)).astype(np.int64), minlength=B * B).astype(np.float64)
    tri = np.bincount((((d1[ok3] + T) * B + (d2[ok3] + T)) * B + (d3[ok3] + T)).astype(np.int64),
                      minlength=B ** 3).astype(np.float64).reshape(B * B, B)
    return (tri / np.maximum(pair, 1.0)[:, None]).ravel()

def spam686(img, T=3):
    x = img.astype(np.int32)
    hv = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    dg = [(1, 1), (-1, -1), (1, -1), (-1, 1)]
    f1 = np.mean([_spam_dir(x, dy, dx, T) for dy, dx in hv], axis=0)
    f2 = np.mean([_spam_dir(x, dy, dx, T) for dy, dx in dg], axis=0)
    return np.concatenate([f1, f2])

FEATURIZERS = {"F13": feat13, "SPAM": spam686}

def features(images, name, which):
    f = WORK_DIR / f"feat_{which}_{name}.npy"
    if f.exists():
        return np.load(f)
    fn = FEATURIZERS[which]
    t0 = time.time()
    out = np.stack(Parallel(n_jobs=CONFIG["n_jobs"], batch_size=64)(
        delayed(fn)(np.asarray(images[i])) for i in range(len(images))))
    np.save(f, out)
    log(f"  features {which}/{name}: {out.shape} ({time.time() - t0:.0f}s)")
    return out

# sanity: on a horizontal ramp every direction has one occupied context with a deterministic transition
_t = np.tile(np.arange(40, dtype=np.uint8), (40, 1))
_f = spam686(_t)
assert _f.shape == (686,) and abs(_f[:343].sum() - 1.0) < 1e-12 and abs(_f[:343].max() - 0.5) < 1e-12
assert abs(_f[171] - 0.5) < 1e-12   # context (0, 0) -> 0 from the two vertical directions

FEAT = {w: {} for w in FEATURIZERS}
log("Extracting features")
for w in FEATURIZERS:
    FEAT[w]["cover"] = features(COVERS, "cover", w)
    FEAT[w]["cover_shift"] = features(COVERS_SHIFT, "cover_shift", w)
    for k in STEGO:
        FEAT[w][k] = features(STEGO[k], k, w)
        FEAT[w]["shift_" + k] = features(STEGO_SHIFT[k], "shift_" + k, w)
    for (q, a), arr in CTRL.items():
        FEAT[w][f"ctrl_q{q:.2f}_a{a:.2f}"] = features(arr, f"ctrl_q{q:.2f}_a{a:.2f}", w)
log("Feature extraction done.")

## 5. Detectors, metrics and resampling

AUC is threshold-free. The error $P_E = (P_{FA} + P_{MD})/2$ uses a threshold selected on validation covers. Confidence intervals come from a percentile bootstrap over test covers, so a cover and its stego are always resampled together.

In [ ]:
# Detectors on features, metrics, and bootstrap over covers
def auc_score(s0, s1):
    """Area under the ROC curve from cover scores s0 and stego scores s1 (ties count one half)."""
    r = stats.rankdata(np.concatenate([s0, s1]))
    n0, n1 = len(s0), len(s1)
    return float((r[n0:].sum() - n1 * (n1 + 1) / 2.0) / (n0 * n1))

def best_threshold(s0, s1):
    """Threshold minimising (P_FA + P_MD) / 2. Used on validation scores only."""
    c = np.sort(np.unique(np.concatenate([s0, s1])))
    if len(c) > 2000:
        c = np.quantile(c, np.linspace(0, 1, 2000))
    pe = [0.5 * (np.mean(s0 > t) + np.mean(s1 <= t)) for t in c]
    return float(c[int(np.argmin(pe))])

def pe_at(s0, s1, thr):
    return float(0.5 * (np.mean(s0 > thr) + np.mean(s1 <= thr)))

class FLDEnsemble:
    """Fisher linear discriminants on random feature subspaces, trained on bootstrap samples of pairs.
    A fixed-size simplification of the ensemble classifier used with rich features."""
    def __init__(self, n_learners=51, d_sub=200, ridge=1e-6, seed=0):
        self.L, self.d, self.ridge, self.seed = n_learners, d_sub, ridge, seed
    def fit(self, X0, X1):
        rng = np.random.default_rng(self.seed)
        self.mu, self.sd = X0.mean(0), X0.std(0) + 1e-12
        Z0, Z1 = (X0 - self.mu) / self.sd, (X1 - self.mu) / self.sd
        n, D = Z0.shape
        d = min(self.d, D)
        self.learners = []
        for _ in range(self.L):
            cols = rng.choice(D, d, replace=False)
            b = rng.integers(0, n, n)
            A0, A1 = Z0[b][:, cols], Z1[b][:, cols]
            m0, m1 = A0.mean(0), A1.mean(0)
            Sw = np.cov(A0, rowvar=False) + np.cov(A1, rowvar=False)
            Sw = np.atleast_2d(Sw) + self.ridge * np.trace(np.atleast_2d(Sw)) / d * np.eye(d)
            w = np.linalg.solve(Sw, m1 - m0)
            p0, p1 = A0 @ w, A1 @ w
            self.learners.append((cols, w, 0.5 * (p0.mean() + p1.mean()), np.sqrt(0.5 * (p0.var() + p1.var())) + 1e-12))
        return self
    def score(self, X):
        Z = (X - self.mu) / self.sd
        return np.mean([(Z[:, c] @ w - t) / s for c, w, t, s in self.learners], axis=0)

def make_detector(name, seed=0):
    if name == "LR":
        m = Pipeline([("sc", StandardScaler()), ("clf", LogisticRegression(max_iter=2000, random_state=seed))])
    elif name == "RF":
        m = RandomForestClassifier(random_state=seed, max_features="sqrt", n_jobs=CONFIG["n_jobs"], **CONFIG["rf"])
    elif name == "FLD":
        return FLDEnsemble(seed=seed, **CONFIG["fld_ensemble"])
    else:
        raise ValueError(name)
    class _W:
        def fit(self, X0, X1):
            m.fit(np.vstack([X0, X1]), np.r_[np.zeros(len(X0)), np.ones(len(X1))]); return self
        def score(self, X):
            return m.predict_proba(X)[:, 1]
    return _W()

def boot_ci(fn, n, n_boot=None, seed=0, level=None):
    """Percentile bootstrap over covers: fn(idx) is evaluated on resampled cover indices."""
    n_boot = n_boot or CONFIG["n_boot"]; level = level or CONFIG["ci_level"]
    rng = np.random.default_rng(seed)
    v = np.array([fn(rng.integers(0, n, n)) for _ in range(n_boot)])
    a = (1 - level) / 2
    return np.quantile(v, [a, 1 - a], axis=0)

def evaluate_scores(v0, v1, t0, t1, seed=0):
    """v*: validation scores (threshold selection); t*: test scores, aligned by cover."""
    thr = best_threshold(v0, v1)
    n = len(t0)
    ci = boot_ci(lambda b: np.array([auc_score(t0[b], t1[b]), pe_at(t0[b], t1[b], thr)]), n, seed=seed)
    return {"auc": auc_score(t0, t1), "auc_lo": float(ci[0, 0]), "auc_hi": float(ci[1, 0]),
            "pe": pe_at(t0, t1, thr), "pe_lo": float(ci[0, 1]), "pe_hi": float(ci[1, 1]), "n_test": int(n)}

DETECTORS = [("LR", "F13"), ("RF", "F13"), ("FLD", "SPAM")]   # (classifier, feature space)
SCORES = {}     # test scores kept for paired comparisons: SCORES[(det, protocol, condition)] = (t0, t1)

def run_feature_detector(det, space, F0, F1, tr, va, te, F0_test=None, F1_test=None, seed=0):
    m = make_detector(det, seed).fit(F0[tr], F1[tr])
    v0, v1 = m.score(F0[va]), m.score(F1[va])
    if F0_test is None:
        t0, t1 = m.score(F0[te]), m.score(F1[te])
    else:
        t0, t1 = m.score(F0_test), m.score(F1_test)
    return evaluate_scores(v0, v1, t0, t1, seed), (t0, t1)

def row_level_split(n, seed):
    """Leaky split (P1): rows are assigned to partitions individually, so the cover and the stego
    of one source image can fall on different sides. Sizes match the source-disjoint split."""
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(2):  # class 0 rows, class 1 rows
        p = rng.permutation(n)
        out.append((p[:len(IDX["train"])], p[len(IDX["train"]):len(IDX["train"]) + len(IDX["val"])],
                    p[len(IDX["train"]) + len(IDX["val"]):]))
    return out

def cross_overlap(split0, split1):
    """Share of test rows whose pair partner sits in the training partition."""
    (tr0, _, te0), (tr1, _, te1) = split0, split1
    return float((np.isin(te0, tr1).mean() + np.isin(te1, tr0).mean()) / 2)

## 6. Experiment A — feature detectors under protocols P1, P2 and P3

In [ ]:
# Experiment A: detection under source-disjoint (P2), row-level (P1) and source-shifted (P3) protocols
rows = []
t_start = time.time()
tr, va, te = IDX["train"], IDX["val"], IDX["test"]
s0, s1 = row_level_split(N_TOTAL, CONFIG["seed"])
RESULTS["p1_pair_crossing_rate"] = cross_overlap(s0, s1)
log(f"P1 row-level split: {RESULTS['p1_pair_crossing_rate']:.3f} of test rows have their pair partner in training")
for ci, (emb, p) in enumerate(CONDITIONS):
    k = cond_key(emb, p)
    for det, space in DETECTORS:
        F0, F1 = FEAT[space]["cover"], FEAT[space][k]
        # P2: every cover and its stego stay in one partition
        r, sc = run_feature_detector(det, space, F0, F1, tr, va, te)
        SCORES[(det, "P2", k)] = sc
        rows.append({"embedder": emb, "payload": p, "detector": f"{det}-{space}", "protocol": "P2", **r})
        # P1: rows split individually
        m = make_detector(det).fit(F0[s0[0]], F1[s1[0]])
        v0, v1, t0, t1 = m.score(F0[s0[1]]), m.score(F1[s1[1]]), m.score(F0[s0[2]]), m.score(F1[s1[2]])
        thr = best_threshold(v0, v1)
        rng = np.random.default_rng(1)
        bs = np.array([[auc_score(t0[b0], t1[b1]), pe_at(t0[b0], t1[b1], thr)]
                       for b0, b1 in ((rng.integers(0, len(t0), len(t0)), rng.integers(0, len(t1), len(t1)))
                                      for _ in range(CONFIG["n_boot"]))])
        lo, hi = np.quantile(bs, [0.025, 0.975], axis=0)
        rows.append({"embedder": emb, "payload": p, "detector": f"{det}-{space}", "protocol": "P1",
                     "auc": auc_score(t0, t1), "auc_lo": lo[0], "auc_hi": hi[0],
                     "pe": pe_at(t0, t1, thr), "pe_lo": lo[1], "pe_hi": hi[1], "n_test": len(t0)})
        # P3: trained as P2, tested on the shifted source (same test images, different processing)
        r, sc = run_feature_detector(det, space, F0, F1, tr, va, te,
                                     FEAT[space]["cover_shift"], FEAT[space]["shift_" + k])
        SCORES[(det, "P3", k)] = sc
        rows.append({"embedder": emb, "payload": p, "detector": f"{det}-{space}", "protocol": "P3", **r})
    log(f"  [{ci + 1}/{len(CONDITIONS)}] {k} done ({time.time() - t_start:.0f}s)")
DET = pd.DataFrame(rows)
DET.to_csv(OUT_DIR / "detection_feature_detectors.csv", index=False)
log("Feature detectors, P2 (source-disjoint):\n" +
    DET[DET.protocol == "P2"].pivot_table(index=["embedder", "payload"], columns="detector", values="pe").round(4).to_string())

# P1 diagnosis: test rows whose pair partner was seen in training with the opposite label, against the rest.
c0, c1 = np.isin(s0[2], s1[0]), np.isin(s1[2], s0[0])
brk = []
for emb, p in CONDITIONS:
    k = cond_key(emb, p)
    for det, space in DETECTORS:
        F0, F1 = FEAT[space]["cover"], FEAT[space][k]
        m = make_detector(det).fit(F0[s0[0]], F1[s1[0]])
        t0, t1 = m.score(F0[s0[2]]), m.score(F1[s1[2]])
        brk.append({"embedder": emb, "payload": p, "detector": f"{det}-{space}", "auc_all": auc_score(t0, t1),
                    "auc_partner_in_train": auc_score(t0[c0], t1[c1]),
                    "auc_partner_not_in_train": auc_score(t0[~c0], t1[~c1]),
                    "n_partner_in_train": int(c0.sum() + c1.sum()), "n_partner_not_in_train": int((~c0).sum() + (~c1).sum())})
    log(f"  P1 breakdown {k} done")
BRK = pd.DataFrame(brk)
BRK.to_csv(OUT_DIR / "p1_pair_breakdown.csv", index=False)
log("P1 breakdown (mean AUC over conditions):\n" +
    BRK.groupby("detector")[["auc_all", "auc_partner_in_train", "auc_partner_not_in_train"]].mean().round(3).to_string())

## 7. Experiment A — SRNet

One network per embedder and payload. Training starts at the highest payload and is continued to lower payloads from the previous weights. Stopping rule: after a warm-up, the learning rate is divided by ten when the validation error has not improved by more than `min_delta` for `patience_lr` epochs, and training stops after a further `patience_stop` epochs without such an improvement, or at the epoch cap.

In [ ]:
# Experiment A (continued): SRNet trained per condition with a payload curriculum, early stopping and resume
SR = CONFIG["srnet"]
SR["enabled"] = SR["enabled"] and os.environ.get("CSS_SRNET", "1") == "1"   # CSS_SRNET=0 skips SRNet
try:
    import torch, torch.nn as nn, torch.nn.functional as Fnn
    HAVE_TORCH = True
except Exception as e:
    HAVE_TORCH = False
    log(f"PyTorch not available ({e}); SRNet is skipped.")

if HAVE_TORCH:
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"SRNet device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

    def _cbr(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
    def _cb(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o))

    class _Res(nn.Module):           # layers 3-7: residual, no pooling
        def __init__(self, c):
            super().__init__(); self.f = nn.Sequential(_cbr(c, c), _cb(c, c))
        def forward(self, x):
            return x + self.f(x)

    class _Down(nn.Module):          # layers 8-11: residual with 3x3 average pooling, stride 2
        def __init__(self, i, o):
            super().__init__()
            self.f = nn.Sequential(_cbr(i, o), _cb(o, o), nn.AvgPool2d(3, 2, 1))
            self.s = nn.Sequential(nn.Conv2d(i, o, 1, stride=2, bias=False), nn.BatchNorm2d(o))
        def forward(self, x):
            return self.s(x) + self.f(x)

    class SRNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.front = nn.Sequential(_cbr(1, 64), _cbr(64, 16), *[_Res(16) for _ in range(5)])
            self.mid = nn.Sequential(_Down(16, 16), _Down(16, 64), _Down(64, 128), _Down(128, 256))
            self.last = nn.Sequential(_cbr(256, 512), _cb(512, 512))
            self.fc = nn.Linear(512, 2)
        def forward(self, x):
            x = self.last(self.mid(self.front(x)))
            return self.fc(x.mean(dim=(2, 3)))

    def _to_dev(arr, idx):
        return torch.from_numpy(np.ascontiguousarray(np.asarray(arr)[np.sort(idx)])).to(DEVICE)

    @torch.no_grad()
    def srnet_scores(model, x_u8, bs=64):
        model.eval(); out = []
        for s in range(0, len(x_u8), bs):
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                lg = model(x_u8[s:s + bs].float().unsqueeze(1))
            out.append((lg[:, 1] - lg[:, 0]).float().cpu().numpy())
        return np.concatenate(out)

    def _augment(c, s):
        """Same random rotation/flip for the cover and the stego of a pair."""
        k = int(torch.randint(0, 4, (1,)))
        c, s = torch.rot90(c, k, (1, 2)), torch.rot90(s, k, (1, 2))
        if torch.rand(1) < 0.5:
            c, s = c.flip(2), s.flip(2)
        return c, s

    def train_srnet(key, stego, init_from=None, transfer=False):
        ck = WORK_DIR / f"srnet_{key}.pt"
        done = WORK_DIR / f"srnet_{key}.done.json"
        model = SRNet().to(DEVICE)
        if done.exists():
            model.load_state_dict(torch.load(ck, map_location=DEVICE)["best"])
            return model, json.load(open(done))
        c_tr, s_tr = _to_dev(COVERS, IDX["train"]), _to_dev(stego, IDX["train"])
        c_va, s_va = _to_dev(COVERS, IDX["val"]), _to_dev(stego, IDX["val"])
        decay = [p for n_, p in model.named_parameters() if p.ndim == 4]
        other = [p for n_, p in model.named_parameters() if p.ndim != 4]
        opt = torch.optim.Adamax([{"params": decay, "weight_decay": SR["weight_decay"]},
                                  {"params": other, "weight_decay": 0.0}], lr=SR["lr"])
        try:
            scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
        except AttributeError:
            scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")
        # SRNet often stays at chance for many epochs before it starts to converge: the stopping
        # counters only run after a warm-up, or as soon as the validation error has left chance level.
        # A model carried over from another embedding scheme gets the full warm-up and epoch budget.
        fresh = init_from is None or transfer
        warm = SR["warmup_epochs"] if fresh else 0
        st = {"epoch": 0, "best_err": 1.0, "since": 0, "lr_dropped": False, "history": []}
        max_ep = SR["max_epochs_scratch"] if fresh else SR["max_epochs_finetune"]
        if ck.exists():                                   # resume an interrupted run
            z = torch.load(ck, map_location=DEVICE)
            model.load_state_dict(z["last"]); opt.load_state_dict(z["opt"]); st = z["state"]; best = z["best"]
            log(f"  {key}: resuming at epoch {st['epoch']}")
        else:
            if init_from is not None:
                model.load_state_dict(init_from)
            best = {k_: v.clone() for k_, v in model.state_dict().items()}
        n, bp = len(c_tr), SR["batch_pairs"]
        gen = torch.Generator(device="cpu").manual_seed(CONFIG["seed"] + st["epoch"])
        stop_reason = "max_epochs"
        while st["epoch"] < max_ep:
            model.train(); t0 = time.time(); tot = 0.0
            perm = torch.randperm(n, generator=gen)
            for b in range(0, n - bp + 1, bp):
                ii = perm[b:b + bp].to(DEVICE)
                c, s = _augment(c_tr[ii], s_tr[ii])
                x = torch.cat([c, s]).float().unsqueeze(1)
                y = torch.cat([torch.zeros(bp), torch.ones(bp)]).long().to(DEVICE)
                with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                    loss = Fnn.cross_entropy(model(x), y)
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                tot += float(loss.detach())
            v0, v1 = srnet_scores(model, c_va), srnet_scores(model, s_va)
            err = 0.5 * (np.mean(v0 > 0) + np.mean(v1 <= 0))
            st["epoch"] += 1
            improved = err < st["best_err"] - SR["min_delta"]
            if improved:
                st["best_err"], st["since"] = float(err), 0
                best = {k_: v.clone() for k_, v in model.state_dict().items()}
            elif st["epoch"] > warm or st["best_err"] < 0.47:
                st["since"] += 1
            st["history"].append([st["epoch"], tot / max(1, n // bp), float(err)])
            if st["epoch"] % 5 == 0 or st["epoch"] == 1 or improved:
                log(f"  {key}: epoch {st['epoch']:3d}/{max_ep} loss {tot / max(1, n // bp):.4f} "
                    f"val_err {err:.4f} best {st['best_err']:.4f} idle {st['since']} ({time.time() - t0:.0f}s/epoch)")
            if not st["lr_dropped"] and st["since"] >= SR["patience_lr"]:
                for g in opt.param_groups:
                    g["lr"] = SR["lr"] * 0.1
                st["lr_dropped"], st["since"] = True, 0
                model.load_state_dict(best)
                log(f"  {key}: no gain above {SR['min_delta']} for {SR['patience_lr']} epochs -> learning rate x0.1")
            elif st["lr_dropped"] and st["since"] >= SR["patience_stop"]:
                stop_reason = "no_significant_improvement"; break
            torch.save({"last": model.state_dict(), "best": best, "opt": opt.state_dict(), "state": st}, ck)
        torch.save({"last": model.state_dict(), "best": best, "opt": opt.state_dict(), "state": st}, ck)
        info = {"epochs": st["epoch"], "best_val_err": st["best_err"], "stop": stop_reason,
                "converged": bool(st["best_err"] < 0.49), "init": "transfer" if transfer else ("curriculum" if init_from is not None else "scratch")}
        json.dump(info, open(done, "w"))
        log(f"  {key}: stopped after {st['epoch']} epochs ({stop_reason}); best val_err {st['best_err']:.4f}")
        model.load_state_dict(best)
        del c_tr, s_tr
        return model, info

    cnn_rows, SR_INFO = [], {}
    if SR["enabled"]:
        c_va, c_te = _to_dev(COVERS, IDX["val"]), _to_dev(COVERS, IDX["test"])
        c_sh = torch.from_numpy(np.array(COVERS_SHIFT)).to(DEVICE)
        # SRNet trained from scratch on 4,000 pairs does not leave chance level for the content-adaptive
        # schemes, so HILL and S-UNIWARD start from the LSBM-A model at the highest payload (LSBM-A also
        # concentrates changes in textured regions); lower payloads follow the payload curriculum.
        TRANSFER_FROM = {"HILL": "LSBM-A", "S-UNIWARD": "LSBM-A"}
        top = {}
        for emb in CONFIG["embedders"]:
            src = TRANSFER_FROM.get(emb)
            prev = top.get(src)
            for i, p in enumerate(SR["payloads_bpp"]):
                k = cond_key(emb, p)
                if k not in STEGO:
                    continue
                model, info = train_srnet(k, STEGO[k], init_from=prev, transfer=(i == 0 and prev is not None))
                SR_INFO[k] = info
                state = {k_: v.clone() for k_, v in model.state_dict().items()}
                if i == 0:
                    top[emb] = state
                # only a model that has clearly left chance level is carried to the next payload
                prev = state if info["best_val_err"] < 0.45 else prev
                v0, v1 = srnet_scores(model, c_va), srnet_scores(model, _to_dev(STEGO[k], IDX["val"]))
                t0_, t1_ = srnet_scores(model, c_te), srnet_scores(model, _to_dev(STEGO[k], IDX["test"]))
                SCORES[("SRNet", "P2", k)] = (t0_, t1_)
                cnn_rows.append({"embedder": emb, "payload": p, "detector": "SRNet", "protocol": "P2",
                                 **evaluate_scores(v0, v1, t0_, t1_)})
                h0 = srnet_scores(model, c_sh)
                h1 = srnet_scores(model, torch.from_numpy(np.array(STEGO_SHIFT[k])).to(DEVICE))
                SCORES[("SRNet", "P3", k)] = (h0, h1)
                cnn_rows.append({"embedder": emb, "payload": p, "detector": "SRNet", "protocol": "P3",
                                 **evaluate_scores(v0, v1, h0, h1)})
                log(f"  SRNet {k}: P_E(P2)={cnn_rows[-2]['pe']:.4f}  P_E(P3)={cnn_rows[-1]['pe']:.4f}")
        if cnn_rows:
            DET = pd.concat([DET[DET.detector != "SRNet"], pd.DataFrame(cnn_rows)], ignore_index=True)
            DET.to_csv(OUT_DIR / "detection_all_detectors.csv", index=False)
        RESULTS["srnet_training"] = SR_INFO
        save_results()

## 8. Experiment B — separability measures as predictors of detector error

Measures are computed on training covers only. Their agreement with detector error on test covers is summarised by Spearman's rank correlation over all conditions and by Kendall's τ over embedders at fixed payload.

In [ ]:
# Experiment B: do separability measures computed on training features predict detector error on test covers?
def _std(F0, F1):
    mu, sd = F0.mean(0), F0.std(0) + 1e-12
    return (F0 - mu) / sd, (F1 - mu) / sd

def m_fisher_mean(Z0, Z1):
    return float(np.mean((Z0.mean(0) - Z1.mean(0)) ** 2 / (Z0.var(0) + Z1.var(0) + 1e-12)))

def m_fisher_max(Z0, Z1):
    return float(np.max((Z0.mean(0) - Z1.mean(0)) ** 2 / (Z0.var(0) + Z1.var(0) + 1e-12)))

def m_mean_shift(Z0, Z1):
    return float(np.linalg.norm(Z0.mean(0) - Z1.mean(0)))

def m_mahalanobis(Z0, Z1, shrink=None):
    """Squared deflection d^2 = dmu' (S + shrink * tr(S)/d * I)^-1 dmu with the pooled covariance S."""
    shrink = CONFIG["shrinkage"] if shrink is None else shrink
    d = Z0.shape[1]
    S = 0.5 * (np.cov(Z0, rowvar=False) + np.cov(Z1, rowvar=False))
    S = S + shrink * np.trace(S) / d * np.eye(d)
    dm = Z1.mean(0) - Z0.mean(0)
    return float(dm @ np.linalg.solve(S, dm))

def m_paired(Z0, Z1, shrink=None):
    """Same form on within-pair differences: uses the cover/stego pairing, which a warden does not have."""
    shrink = CONFIG["shrinkage"] if shrink is None else shrink
    Dm = Z1 - Z0; d = Dm.shape[1]
    S = np.cov(Dm, rowvar=False); S = S + shrink * np.trace(S) / d * np.eye(d)
    dm = Dm.mean(0)
    return float(dm @ np.linalg.solve(S, dm))

def m_mmd(Z0, Z1, seed=0):
    """Unbiased squared maximum mean discrepancy, Gaussian kernel, median-distance bandwidth."""
    rng = np.random.default_rng(seed)
    n = min(len(Z0), CONFIG["mmd_subsample"])
    a, b = Z0[rng.choice(len(Z0), n, replace=False)], Z1[rng.choice(len(Z1), n, replace=False)]
    def sq(x, y):
        return np.maximum((x * x).sum(1)[:, None] + (y * y).sum(1)[None, :] - 2 * x @ y.T, 0.0)
    daa, dbb, dab = sq(a, a), sq(b, b), sq(a, b)
    g = 1.0 / max(np.median(dab), 1e-12)
    kaa, kbb, kab = np.exp(-g * daa), np.exp(-g * dbb), np.exp(-g * dab)
    np.fill_diagonal(kaa, 0); np.fill_diagonal(kbb, 0)
    return float(kaa.sum() / (n * (n - 1)) + kbb.sum() / (n * (n - 1)) - 2 * kab.mean())

def m_js(Z0, Z1, bins=20):
    """Mean per-feature Jensen-Shannon divergence on pooled-quantile bins."""
    out = []
    for j in range(Z0.shape[1]):
        e = np.unique(np.quantile(np.r_[Z0[:, j], Z1[:, j]], np.linspace(0, 1, bins + 1)))
        if len(e) < 3:
            out.append(0.0); continue
        p = np.histogram(Z0[:, j], e)[0] + 1e-9; q = np.histogram(Z1[:, j], e)[0] + 1e-9
        p, q = p / p.sum(), q / q.sum(); m = 0.5 * (p + q)
        out.append(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))
    return float(np.mean(out))

MEASURES = {"Fisher (mean)": m_fisher_mean, "Fisher (max)": m_fisher_max, "Mean shift": m_mean_shift,
            "Deflection": m_mahalanobis, "Paired deflection": m_paired, "MMD": m_mmd, "JS (mean)": m_js}

mrows = []
t0 = time.time()
for ci, (emb, p) in enumerate(CONDITIONS):
    k = cond_key(emb, p)
    for space in FEATURIZERS:
        Z0, Z1 = _std(FEAT[space]["cover"][IDX["train"]], FEAT[space][k][IDX["train"]])
        for name, fn in MEASURES.items():
            mrows.append({"embedder": emb, "payload": p, "space": space, "measure": name, "value": fn(Z0, Z1)})
    log(f"  measures [{ci + 1}/{len(CONDITIONS)}] {k} ({time.time() - t0:.0f}s)")
MEAS = pd.DataFrame(mrows)
MEAS.to_csv(OUT_DIR / "separability_measures.csv", index=False)

# Rank agreement between each measure (training covers) and each detector's error (test covers, P2).
P2 = DET[DET.protocol == "P2"]
prow = []
rng = np.random.default_rng(CONFIG["seed"])
for (space, mname), g in MEAS.groupby(["space", "measure"]):
    for det, h in P2.groupby("detector"):
        j = g.merge(h, on=["embedder", "payload"])
        if len(j) < 4:
            continue
        if j["pe"].nunique() < 2:
            continue
        rho = stats.spearmanr(j["value"], j["pe"]).statistic
        bs = []
        for _ in range(CONFIG["n_boot"]):
            b = rng.integers(0, len(j), len(j))
            if len(np.unique(j["value"].values[b])) > 2 and len(np.unique(j["pe"].values[b])) > 1:
                bs.append(stats.spearmanr(j["value"].values[b], j["pe"].values[b]).statistic)
        bs = [v for v in bs if np.isfinite(v)]
        lo, hi = np.quantile(bs, [0.025, 0.975]) if bs else (np.nan, np.nan)
        # ordering of embedders at a fixed payload (does the measure rank schemes like the detector does?)
        taus = [stats.kendalltau(q["value"], q["pe"]).statistic for _, q in j.groupby("payload") if len(q) > 2]
        taus = [t for t in taus if np.isfinite(t)]
        prow.append({"space": space, "measure": mname, "detector": det, "spearman": rho, "lo": lo, "hi": hi,
                     "kendall_within_payload": float(np.nanmean(taus)) if taus else np.nan, "n_conditions": len(j)})
PRED = pd.DataFrame(prow)
PRED.to_csv(OUT_DIR / "predictor_rank_agreement.csv", index=False)
log("Spearman correlation between measure and P_E (negative = the measure tracks detectability):\n" +
    PRED.pivot_table(index=["space", "measure"], columns="detector", values="spearman").round(3).to_string())

# Baseline predictor: the embedding change rate, which needs no feature space at all.
brow = []
rng_b = np.random.default_rng(CONFIG["seed"] + 9)
for det, h in P2.groupby("detector"):
    j = h.assign(value=[RESULTS["change_rate"][cond_key(e, q)] for e, q in zip(h["embedder"], h["payload"])])
    rho = stats.spearmanr(j["value"], j["pe"]).statistic
    bs = []
    for _ in range(CONFIG["n_boot"]):
        b = rng_b.integers(0, len(j), len(j))
        if len(np.unique(j["value"].values[b])) > 2 and len(np.unique(j["pe"].values[b])) > 1:
            bs.append(stats.spearmanr(j["value"].values[b], j["pe"].values[b]).statistic)
    bs = [v for v in bs if np.isfinite(v)]
    lo, hi = np.quantile(bs, [0.025, 0.975]) if bs else (np.nan, np.nan)
    taus = [stats.kendalltau(q["value"], q["pe"]).statistic for _, q in j.groupby("payload") if len(q) > 2]
    taus = [t for t in taus if np.isfinite(t)]
    brow.append({"detector": det, "spearman": rho, "lo": lo, "hi": hi,
                 "kendall_within_payload": float(np.nanmean(taus)) if taus else np.nan, "n_conditions": len(j)})
BASE = pd.DataFrame(brow)
BASE.to_csv(OUT_DIR / "predictor_baseline_change_rate.csv", index=False)
log("Baseline, change rate against P_E:\n" + BASE.round(3).to_string(index=False))

## 9. Experiment C — whitening as a negative and a positive control

For a uniform message, XOR-masking leaves the message uniform, so the stego distribution does not depend on the whitening level α: any reported difference between whitening levels is sampling variation. For a biased message, full masking removes the bias of the embedded bits, which is a real change. The single-cover protocol P0 is replicated many times to measure how often it singles out a non-zero 'preferred' α under the null.

In [ ]:
# Experiment C: whitening as a negative control (uniform message) and a positive control (biased message)
# C1. Source-disjoint protocol on many covers: difference in AUC between masked and unmasked embedding.
crow = []
tr, va, te = IDX["train"], IDX["val"], IDX["test"]
for q in CONFIG["control"]["bit_bias"]:
    base = {}
    for a in CONFIG["control"]["alphas"]:
        name = f"ctrl_q{q:.2f}_a{a:.2f}"
        for det, space in DETECTORS:
            F0, F1 = FEAT[space]["cover"], FEAT[space][name]
            r, (t0_, t1_) = run_feature_detector(det, space, F0, F1, tr, va, te)
            if a == 0.0:
                base[det] = (t0_, t1_, r["auc"])
                d, lo, hi = 0.0, 0.0, 0.0
            else:
                b0, b1, _ = base[det]
                d = r["auc"] - base[det][2]
                ci = boot_ci(lambda b: auc_score(t0_[b], t1_[b]) - auc_score(b0[b], b1[b]), len(t0_), seed=7)
                lo, hi = float(ci[0]), float(ci[1])
            crow.append({"protocol": "P2", "bit_bias": q, "alpha": a, "detector": f"{det}-{space}",
                         "auc": r["auc"], "delta_auc": d, "lo": lo, "hi": hi,
                         "effect_declared": bool(a > 0 and (lo > 0 or hi < 0))})
    log(f"  control, bit bias {q}: done")
CTRL_P2 = pd.DataFrame(crow)
CTRL_P2.to_csv(OUT_DIR / "control_source_disjoint.csv", index=False)
log("Whitening control under P2:\n" + CTRL_P2.round(4).to_string(index=False))

# C2. Legacy protocol P0: one cover, 36 stego realisations, Gaussian jitter on the features, 30% row split.
# The message is uniform, so any 'preferred' whitening level found here is spurious by construction.
P0 = CONFIG["p0"]
def p0_auc(cover, alpha, seed, payload, det="LR"):
    rng = np.random.default_rng(seed)
    f0 = feat13(cover)
    X0 = f0 + rng.normal(0, P0["jitter_sigma"], (P0["n_variants"], 13))
    X1 = np.stack([feat13(lsbm_embed(cover, payload, np.random.default_rng([seed, i, int(alpha * 100)]),
                                     adaptive=True, alpha=alpha))
                   for i in range(P0["n_variants"])]) + rng.normal(0, P0["jitter_sigma"], (P0["n_variants"], 13))
    nt = int(round(P0["n_variants"] * P0["test_size"]))
    p0_, p1_ = rng.permutation(P0["n_variants"]), rng.permutation(P0["n_variants"])
    m = make_detector(det, seed).fit(X0[p0_[nt:]], X1[p1_[nt:]])
    return auc_score(m.score(X0[p0_[:nt]]), m.score(X1[p1_[:nt]]))

def p0_replicate(cover, ci, rep, payload):
    return [ci, rep, payload] + [p0_auc(cover, a, 1000 * ci + rep, payload) for a in P0["alpha_grid"]]

jobs = [(ci, rep, pl) for ci in range(P0["n_source_covers"]) for rep in range(P0["n_replicates"])
        for pl in [0.10, 0.15]]
log(f"P0 replicates: {len(jobs)} runs x {len(P0['alpha_grid'])} whitening levels")
t0 = time.time(); out = []
for s in range(0, len(jobs), 400):
    out += Parallel(n_jobs=CONFIG["n_jobs"])(delayed(p0_replicate)(np.asarray(COVERS[IDX["test"][j[0]]]), *j) for j in jobs[s:s + 400])
    log(f"  P0 {min(len(jobs), s + 400)}/{len(jobs)} ({time.time() - t0:.0f}s)")
acols = [f"a{a:.2f}" for a in P0["alpha_grid"]]
P0DF = pd.DataFrame(out, columns=["cover", "rep", "payload"] + acols)
P0DF.to_csv(OUT_DIR / "control_legacy_single_cover.csv", index=False)
A = P0DF[acols].values
_r = np.random.default_rng(CONFIG["seed"])
best = (A + 1e-9 * _r.random(A.shape)).argmin(1)      # AUC is discrete on 11+11 test rows: break ties at random
RESULTS["control_p0"] = {
    "n_runs": int(len(A)),
    "rate_preferred_alpha_nonzero": float(np.mean(best != 0)),
    "rate_expected_under_exchangeability": 1 - 1 / len(acols),
    "preferred_alpha_histogram": {c: float(np.mean(best == i)) for i, c in enumerate(acols)},
    "median_auc_range_across_alpha": float(np.median(A.max(1) - A.min(1))),
    "rate_range_above_0.05": float(np.mean(A.max(1) - A.min(1) > 0.05)),
    "mean_auc": float(A.mean()), "sd_auc_between_replicates": float(A.std(0).mean()),
}
RESULTS["control_p2"] = CTRL_P2.to_dict(orient="records")
save_results()
log("P0 control summary: " + json.dumps(RESULTS["control_p0"], indent=1))

## 10. Experiment D — uncertainty versus number of test covers

In [ ]:
# Experiment D: how the uncertainty of an AUC estimate shrinks with the number of independent test covers
srow = []
rng = np.random.default_rng(CONFIG["seed"])
for (det, proto, k), (t0_, t1_) in SCORES.items():
    if proto != "P2":
        continue
    for n in CONFIG["ntest_grid"]:
        n = min(n, len(t0_))        # resampling with replacement: an estimate from n independent test covers
        a = np.array([auc_score(t0_[b], t1_[b]) for b in (rng.integers(0, len(t0_), n) for _ in range(300))])
        srow.append({"detector": det, "condition": k, "n_test": n, "auc_mean": a.mean(), "auc_sd": a.std(ddof=1),
                     "ci95_halfwidth": 1.96 * a.std(ddof=1)})
SS = pd.DataFrame(srow)
SS.to_csv(OUT_DIR / "auc_uncertainty_vs_ntest.csv", index=False)
SSUM = SS.groupby("n_test")["auc_sd"].agg(["median", "max"]).reset_index()
log("Standard deviation of an AUC estimate from n test covers (all detectors and conditions):\n" +
    SSUM.round(4).to_string(index=False))
RESULTS["auc_sd_vs_ntest"] = SSUM.to_dict(orient="records")
save_results()

## 11. Tables and figures

LaTeX rows are written to `OUT_DIR/Tables`, figures to `OUT_DIR/Figures`, and all summary numbers to `OUT_DIR/results.json`.

In [ ]:
# Tables (LaTeX rows) and figures for the manuscript
STAMP = "% QUICK TEST - NOT FOR PUBLICATION\n" if QUICK else ""
def write_tex(name, lines):
    (TABLE_DIR / name).write_text(STAMP + "\n".join(lines) + "\n", encoding="utf-8")
    log(f"  wrote Tables/{name}")

def ci(v, lo, hi, d=3):
    return f"{v:.{d}f} [{lo:.{d}f}, {hi:.{d}f}]"

dets = [d for d in ["LR-F13", "RF-F13", "FLD-SPAM", "SRNet"] if d in set(DET.detector)]
# Table: detection error under the source-disjoint protocol
lines = []
for emb in CONFIG["embedders"]:
    for p in CONFIG["payloads_bpp"]:
        cells = []
        for d in dets:
            r = DET[(DET.protocol == "P2") & (DET.embedder == emb) & (DET.payload == p) & (DET.detector == d)]
            cells.append(ci(r.pe.iloc[0], r.pe_lo.iloc[0], r.pe_hi.iloc[0]) if len(r) else "--")
        lines.append(f"{emb} & {p:.2f} & " + " & ".join(cells) + r" \\")
write_tex("tab_detection.tex", ["% columns: embedder, payload (bpp), " + ", ".join(dets)] + lines)

# Table: protocol effect (AUC), averaged over payloads per embedder
lines = []
for emb in CONFIG["embedders"]:
    for d in dets:
        cells = []
        for pr in ["P1", "P2", "P3"]:
            r = DET[(DET.protocol == pr) & (DET.embedder == emb) & (DET.detector == d)]
            cells.append(f"{r.auc.mean():.3f}" if len(r) else "--")
        lines.append(f"{emb} & {d} & " + " & ".join(cells) + r" \\")
write_tex("tab_protocol.tex", ["% columns: embedder, detector, mean AUC under P1, P2, P3"] + lines)

# Table: rank agreement between measures and detector error
lines = []
for space in FEATURIZERS:
    for mname in MEASURES:
        cells = []
        for d in dets:
            r = PRED[(PRED.space == space) & (PRED.measure == mname) & (PRED.detector == d)]
            cells.append(ci(r.spearman.iloc[0], r.lo.iloc[0], r.hi.iloc[0], 2) if len(r) else "--")
        lines.append(f"{space} & {mname} & " + " & ".join(cells) + r" \\")
write_tex("tab_predictor.tex", ["% columns: feature space, measure, Spearman rho with P_E of " + ", ".join(dets)] + lines)

# Table: whitening control
lines = []
for _, r in CTRL_P2[CTRL_P2.alpha > 0].iterrows():
    lines.append(f"{r.bit_bias:.2f} & {r.alpha:.2f} & {r.detector} & {ci(r.delta_auc, r.lo, r.hi)} & "
                 f"{'yes' if r.effect_declared else 'no'}" + r" \\")
c0 = RESULTS["control_p0"]
lines.append(f"% P0: preferred alpha != 0 in {100 * c0['rate_preferred_alpha_nonzero']:.1f}% of {c0['n_runs']} runs "
             f"(chance level {100 * c0['rate_expected_under_exchangeability']:.0f}%); median AUC range across alpha "
             f"{c0['median_auc_range_across_alpha']:.3f}")
write_tex("tab_control.tex", ["% columns: P(bit=1), alpha, detector, delta AUC vs alpha=0 [95% CI], effect declared"] + lines)

def savefig(name):
    plt.tight_layout()
    for ext in ("pdf", "png"):
        plt.savefig(FIG_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.close()
    log(f"  wrote Figures/{name}.pdf")

mk = {"LSBM": "o", "LSBM-A": "s", "HILL": "^", "S-UNIWARD": "D"}
fig, axes = plt.subplots(1, len(dets), figsize=(3.2 * len(dets), 3.0), sharey=True, squeeze=False)
for ax, d in zip(axes[0], dets):
    for emb in CONFIG["embedders"]:
        r = DET[(DET.protocol == "P2") & (DET.detector == d) & (DET.embedder == emb)].sort_values("payload")
        if len(r):
            ax.errorbar(r.payload, r.pe, yerr=[r.pe - r.pe_lo, r.pe_hi - r.pe], marker=mk[emb], ms=4, capsize=2, label=emb)
    ax.set_title(d, fontsize=9); ax.set_xlabel("Payload (bpp)"); ax.axhline(0.5, ls=":", c="gray", lw=0.8)
axes[0][0].set_ylabel(r"$P_E$ on test covers"); axes[0][0].legend(fontsize=7)
savefig("fig_pe_vs_payload")

sel = [("F13", "Fisher (mean)"), ("F13", "Deflection"), ("SPAM", "Deflection"), ("SPAM", "MMD")]
ref = "SRNet" if "SRNet" in dets else dets[-1]
fig, axes = plt.subplots(1, len(sel), figsize=(3.2 * len(sel), 3.0), sharey=True, squeeze=False)
for ax, (space, mname) in zip(axes[0], sel):
    j = MEAS[(MEAS.space == space) & (MEAS.measure == mname)].merge(
        DET[(DET.protocol == "P2") & (DET.detector == ref)], on=["embedder", "payload"])
    for emb in CONFIG["embedders"]:
        q = j[j.embedder == emb]
        ax.scatter(q["value"], q["pe"], marker=mk[emb], s=22, label=emb)
    if (j["value"] > 0).all():
        ax.set_xscale("log")
    ax.set_title(f"{mname} ({space})", fontsize=9); ax.set_xlabel("Measure on training covers")
axes[0][0].set_ylabel(rf"$P_E$ of {ref} on test covers"); axes[0][0].legend(fontsize=7)
savefig("fig_measure_vs_pe")

fig, ax = plt.subplots(1, 2, figsize=(7.0, 3.0))
rng_ = A.max(1) - A.min(1)
ax[0].hist(rng_, bins=30); ax[0].set_xlabel("AUC range across whitening levels (P0, uniform message)")
ax[0].set_ylabel("Runs")
ax[1].bar(range(len(acols)), [RESULTS["control_p0"]["preferred_alpha_histogram"][c] for c in acols])
ax[1].axhline(1 / len(acols), ls=":", c="gray"); ax[1].set_xticks(range(len(acols)))
ax[1].set_xticklabels([c[1:] for c in acols]); ax[1].set_xlabel(r"'Preferred' $\alpha$ (lowest AUC)")
ax[1].set_ylabel("Share of runs")
savefig("fig_control_legacy")

fig, ax = plt.subplots(figsize=(3.6, 3.0))
for d, g in SS.groupby("detector"):
    h = g.groupby("n_test")["auc_sd"].median()
    ax.plot(h.index, h.values, marker="o", ms=4, label=d)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("Independent test covers")
ax.set_ylabel("Std. dev. of AUC estimate"); ax.legend(fontsize=7)
savefig("fig_auc_sd_vs_ntest")

RESULTS["config"] = CONFIG
RESULTS["artifacts"] = {"tables": sorted(p.name for p in TABLE_DIR.glob("*.tex")),
                        "figures": sorted(p.name for p in FIG_DIR.glob("*.pdf")),
                        "csv": sorted(p.name for p in OUT_DIR.glob("*.csv"))}
save_results()
log("All outputs written to " + str(OUT_DIR))